# 🎯 4. Closures

A **closure** is a function that remembers variables from its enclosing scope, even after that scope has finished executing.

Closures are the **mechanism** that makes decorators work.

| ☕ Java | 🐍 Python |
|---------|----------|
| Lambda captures effectively-final variables | Functions capture **any** variables |
| Can't modify captured variables | `nonlocal` keyword allows mutation |
| Anonymous inner classes | Closures are regular functions with memory |

In [ ]:
from typing import Callable
from functools import partial

## 4.1 What is a Closure?

When an inner function references a variable from its enclosing function, and the outer function returns the inner function — that's a closure.

In [ ]:
def make_greeter(greeting: str) -> Callable[[str], str]:
    """Outer function — creates the closure."""
    
    def greeter(name: str) -> str:
        """Inner function — the closure itself."""
        return f"{greeting}, {name}!"   # 'greeting' is captured
    
    return greeter   # Return the inner function

# Create two closures with different captured values
say_hello = make_greeter("Hello")
say_hola = make_greeter("Hola")

print(say_hello("Alice"))   # Hello, Alice!
print(say_hola("Bob"))      # Hola, Bob!

# The outer function finished, but 'greeting' is still alive!

## 4.2 Factory Functions

Closures are perfect for creating **families of functions** from a template.

In [ ]:
def make_multiplier(factor: int) -> Callable[[int], int]:
    """Factory — creates multiplier functions."""
    def multiplier(x: int) -> int:
        return x * factor   # 'factor' is captured
    return multiplier

double = make_multiplier(2)
triple = make_multiplier(3)
times_ten = make_multiplier(10)

print(f"double(5) = {double(5)}")
print(f"triple(5) = {triple(5)}")
print(f"times_ten(5) = {times_ten(5)}")

# --- Power function factory ---

def make_power(exp: int) -> Callable[[int], int]:
    def power(base: int) -> int:
        return base ** exp
    return power

square = make_power(2)
cube = make_power(3)
print(f"\nsquare(4) = {square(4)}")
print(f"cube(4) = {cube(4)}")

## 4.3 `nonlocal` — Modifying Captured Variables

By default, you can **read** captured variables but not **reassign** them.
Without `nonlocal`, Python treats the assignment as a **new local variable**.

| Keyword | Scope |
|---------|-------|
| `global` | Module-level variable |
| `nonlocal` | Enclosing function variable |

In [ ]:
# ❌ What happens WITHOUT nonlocal?
def make_counter_broken():
    count = 0
    def counter():
        count += 1   # UnboundLocalError!
        return count
    return counter

broken = make_counter_broken()
try:
    broken()
except UnboundLocalError as e:
    print(f"❌ {e}")
    print("   Python sees 'count += 1' as assignment → treats 'count' as local")
    print("   But local 'count' hasn't been assigned yet → UnboundLocalError")

In [ ]:
# ✅ Fixed with nonlocal
def make_counter(start: int = 0) -> Callable[[], int]:
    """Create a counter with mutable state."""
    count = start
    
    def counter() -> int:
        nonlocal count   # Required to modify 'count'
        count += 1
        return count
    
    return counter

counter = make_counter(10)
print(f"counter() = {counter()}")   # 11
print(f"counter() = {counter()}")   # 12
print(f"counter() = {counter()}")   # 13

# Each closure has its own state
counter2 = make_counter()
print(f"\ncounter2() = {counter2()}")   # 1
print(f"counter() = {counter()}")       # 14 (separate state)

## 4.4 Closures as Lightweight Objects

Closures can replace simple classes — they're "objects with a single method."

| Approach | Best for |
|----------|----------|
| Closure | Single behavior + state |
| Class | Multiple methods + complex state |

In [ ]:
# Closure approach — returns multiple functions sharing state
def make_account(owner: str, balance: float = 0):
    """Bank account as closures — functions sharing state."""
    current_balance = balance
    
    def deposit(amount: float) -> float:
        nonlocal current_balance
        current_balance += amount
        return current_balance
    
    def withdraw(amount: float) -> float:
        nonlocal current_balance
        if amount > current_balance:
            raise ValueError(f"Insufficient funds: {current_balance}")
        current_balance -= amount
        return current_balance
    
    def get_balance() -> float:
        return current_balance
    
    def info() -> str:
        return f"{owner}: ${current_balance:.2f}"
    
    return deposit, withdraw, get_balance, info

deposit, withdraw, balance, info = make_account("Alice", 100)

deposit(50)
withdraw(30)
print(info())   # Alice: $120.00

## 4.5 `functools.partial` — Closure Alternative

`partial` is the stdlib way to create closures that pre-fill arguments.
When your closure just forwards args, `partial` is cleaner.

In [ ]:
# Without partial — manual closure
def make_multiplier_closure(factor):
    def multiplier(x):
        return x * factor
    return multiplier

double_c = make_multiplier_closure(2)

# With partial — one line!
def multiply(x, factor):
    return x * factor

double_p = partial(multiply, factor=2)
triple_p = partial(multiply, factor=3)

print(f"Closure:  double_c(5) = {double_c(5)}")
print(f"Partial:  double_p(5) = {double_p(5)}")
print(f"Partial:  triple_p(5) = {triple_p(5)}")

# partial exposes its internals
print(f"\nPartial func: {double_p.func.__name__}")
print(f"Partial keywords: {double_p.keywords}")

In [ ]:
# Real-world use: pre-configured functions
import json

# Create specialized JSON encoders
pretty_json = partial(json.dumps, indent=2, sort_keys=True)
compact_json = partial(json.dumps, separators=(',', ':'))

data = {"name": "Alice", "age": 30, "active": True}

print("Pretty:")
print(pretty_json(data))
print(f"\nCompact: {compact_json(data)}")

print("\n💡 Use partial when you're just pre-filling arguments")
print("   Use closures when you need custom logic or state")

## 4.6 Under the Hood: `__closure__`

Python stores captured variables in a special `__closure__` attribute.

In [ ]:
def make_adder(n: int):
    def adder(x: int):
        return x + n
    return adder

add_five = make_adder(5)

# Inspect the closure
print(f"Has closure: {add_five.__closure__ is not None}")
print(f"Closure cells: {add_five.__closure__}")
print(f"Captured value: {add_five.__closure__[0].cell_contents}")

# Regular functions don't have closures
def regular_func(x):
    return x + 1

print(f"\nRegular function closure: {regular_func.__closure__}")

## 4.7 ⚠️ Gotcha: Late Binding in Loops

A common bug — closures capture **the variable itself**, not its value at the time of creation.

In [ ]:
# ❌ BUG — all functions return 4!
functions_bad = []
for i in range(5):
    def f():
        return i   # Captures the variable 'i', not its value
    functions_bad.append(f)

print("❌ Bug (late binding):")
print([f() for f in functions_bad])   # [4, 4, 4, 4, 4]

# ✅ FIX 1 — Default argument captures current value
functions_good = []
for i in range(5):
    def f(x=i):   # x=i captures current value of i
        return x
    functions_good.append(f)

print("\n✅ Fix with default arg:")
print([f() for f in functions_good])  # [0, 1, 2, 3, 4]

# ✅ FIX 2 — Factory function
def make_func(value):
    def f():
        return value
    return f

functions_factory = [make_func(i) for i in range(5)]
print("\n✅ Fix with factory:")
print([f() for f in functions_factory])  # [0, 1, 2, 3, 4]

## 4.8 How Closures Enable Decorators

Every decorator is a closure! The `wrapper` function **closes over** `func`.

In [ ]:
from functools import wraps

def timer(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        # 'func' is captured from enclosing scope!
        return func(*args, **kwargs)
    return wrapper

@timer
def greet(name):
    return f"Hello, {name}!"

# Proof: wrapper has a closure holding the original function
print(f"greet's closure: {greet.__closure__}")
print(f"Captured func:   {greet.__closure__[0].cell_contents}")
print(f"Original name:   {greet.__closure__[0].cell_contents.__name__}")

print("\n💡 The wrapper remembers which function to call")
print("   even after timer() has returned — that's a closure!")

## 4.9 Exercises

**Exercise 1:** Write a `make_accumulator(start=0)` closure that:
- Each call **adds** the argument to a running total
- Returns the current total
- Example: `acc = make_accumulator(10)` → `acc(5)` = 15 → `acc(3)` = 18

In [ ]:
# Your solution here


In [ ]:
# ✅ Solution
def make_accumulator(start: float = 0) -> Callable[[float], float]:
    total = start
    
    def accumulate(amount: float) -> float:
        nonlocal total
        total += amount
        return total
    
    return accumulate

acc = make_accumulator(10)
print(f"acc(5) = {acc(5)}")     # 15
print(f"acc(3) = {acc(3)}")     # 18
print(f"acc(-8) = {acc(-8)}")   # 10

# Independent instance
acc2 = make_accumulator()
print(f"\nacc2(100) = {acc2(100)}")  # 100
print(f"acc(1) = {acc(1)}")          # 11 (separate state)

**Exercise 2:** Write a `make_password_checker(min_length, require_digit)` closure factory that returns a validation function.
The validator should return `(True, "OK")` or `(False, "reason")`.

In [ ]:
# Your solution here


In [ ]:
# ✅ Solution
def make_password_checker(
    min_length: int = 8,
    require_digit: bool = True
) -> Callable[[str], tuple[bool, str]]:
    """Factory — creates a password validator with captured rules."""
    
    def check(password: str) -> tuple[bool, str]:
        if len(password) < min_length:
            return False, f"Too short (need {min_length}+, got {len(password)})"
        if require_digit and not any(c.isdigit() for c in password):
            return False, "Must contain at least one digit"
        return True, "OK"
    
    return check

# Strict checker
strict = make_password_checker(min_length=12, require_digit=True)
print(f"strict('hello'):     {strict('hello')}")
print(f"strict('helloworld'): {strict('helloworld!!')}")
print(f"strict('helloworld1'): {strict('helloworld12')}")

# Relaxed checker
relaxed = make_password_checker(min_length=4, require_digit=False)
print(f"\nrelaxed('hi'):   {relaxed('hi')}")
print(f"relaxed('hello'): {relaxed('hello')}")

## 📋 Takeaways

| # | Concept | Key Point |
|---|---------|----------|
| 1 | Closure | Inner function that remembers enclosing scope variables |
| 2 | Factory functions | `make_X()` pattern creates families of functions |
| 3 | `nonlocal` | Required to **modify** (not just read) captured variables |
| 4 | Without `nonlocal` | `count += 1` → `UnboundLocalError` (treated as local) |
| 5 | Closures vs Classes | Closures = lightweight objects with hidden state |
| 6 | `functools.partial` | Stdlib shortcut for closures that just pre-fill args |
| 7 | `__closure__` | Inspect captured values: `func.__closure__[0].cell_contents` |
| 8 | Late binding ⚠️ | Loop closures share the same variable — use default args to fix |
| 9 | Decorators = closures | `wrapper` closes over `func` from the decorator |
| 10 | Java comparison | Like lambda captures, but more powerful with `nonlocal` |